#Task 2: SQL Database + Python Data Analysis

#Objective
Use SQL and Pandas to analyze the supplied employee dataset and report data-quality issues transparently.

#Requirements covered
1. Find the five highest-paid employees.
2. Calculate average salary by department.
3. Identify missing salaries and invalid ages.
4. Calculate total employee records, average salary, and employee counts by department.
5. Explain duplicate-record and missing-value handling.

#Assumptions
- `employee_id` is intended to uniquely identify an employee.
- Salaries must be numeric and non-negative.
- Ages must be whole numbers between 0 and 120.
- Missing values are reported, not silently imputed.
- Duplicate IDs are flagged for review; records are not silently deleted.
- Counts are reported at the record level and, where useful, at the unique-employee level.

**Libraries:** Pandas; Python standard-library `sqlite3` and `pathlib`. No database server is required.

## 1. Load and validate the dataset

Place `employees.csv` in the same folder as this notebook. In Google Colab, if the file is not found, the cell will prompt you to upload it.

In [11]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

REQUIRED_COLUMNS = ["employee_id", "name", "department", "salary", "age"]

def locate_employee_csv():
    candidates = [
        Path("employees.csv"),
        Path("/content/employees.csv"),
        Path("/mnt/data/employees.csv"),
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate

    # Keep Colab upload optional so the notebook also works locally.
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            uploaded_name = next(iter(uploaded))
            return Path(uploaded_name)
    except ImportError:
        pass

    raise FileNotFoundError(
        "employees.csv was not found. Place it beside this notebook "
        "or upload it when prompted in Google Colab."
    )

csv_file = locate_employee_csv()
df = pd.read_csv(csv_file)

missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")

# Convert numeric fields explicitly; invalid text becomes missing and is visible
# in the subsequent data-quality checks rather than causing silent calculations.
df["employee_id"] = pd.to_numeric(df["employee_id"], errors="coerce")
df["salary"] = pd.to_numeric(df["salary"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

print(f"Loaded file: {csv_file}")
print(f"Rows: {len(df)} | Columns: {len(df.columns)}")
display(df.head())

Loaded file: employees.csv
Rows: 200 | Columns: 9


,employee_id,name,department,salary,age,city,employment_type,job_level,years_experience
0,1001,Mira Gupta,Product,89000.0,22.0,Gurugram,Full-time,Junior,6
1,1002,Sai Sengupta,Product,169000.0,55.0,Nagpur,Full-time,Intern,5
2,1003,Sara Patel,Sales,64000.0,53.0,Noida,Part-time,Manager,34
3,1004,Aisha Gupta,Engineering,126000.0,49.0,Mumbai,Full-time,Manager,27
4,1005,Saanvi More,Operations,70000.0,30.0,Hyderabad,Full-time,Senior,6


## 2. Create an in-memory SQLite database

The source CSV remains unchanged. SQLite is used only to execute the required SQL queries.

In [12]:
conn = sqlite3.connect(":memory:")
try:
    df.to_sql("employees", conn, index=False, if_exists="replace")
    print("Employee table created in the in-memory SQLite database.")
except Exception:
    conn.close()
    raise

Employee table created in the in-memory SQLite database.


## 3. Top five highest-paid employees

Rows with missing salaries are excluded. `employee_id` provides deterministic ordering when salaries tie.

In [13]:
top_five_query = """
SELECT employee_id, name, department, salary, age
FROM employees
WHERE salary IS NOT NULL
ORDER BY salary DESC, employee_id ASC
LIMIT 5;
"""

top_five = pd.read_sql_query(top_five_query, conn)
display(top_five)

,employee_id,name,department,salary,age
0,1119,Rahul Ansari,Data Science,221000.0,30.0
1,1024,Ayaan Singh,Data Science,212000.0,49.0
2,1098,Rohan Shaikh,Data Science,202000.0,49.0
3,1032,Saanvi Mukherjee,Engineering,194000.0,-4.0
4,1153,Manav Sengupta,Data Science,194000.0,44.0


## 4. Average salary by department

SQL `AVG()` ignores `NULL` salary values. Departments with no valid salaries would return a `NULL` average.

In [14]:
department_average_query = """
SELECT department,
       AVG(salary) AS average_salary,
       COUNT(salary) AS employees_with_valid_salary,
       COUNT(*) AS employee_records
FROM employees
GROUP BY department
ORDER BY department;
"""

department_averages = pd.read_sql_query(department_average_query, conn)
display(department_averages)

,department,average_salary,employees_with_valid_salary,employee_records
0,Administration,58583.333333,12,12
1,Customer Support,56692.307692,13,13
2,Data Science,155266.666667,15,15
3,Engineering,126050.000000,20,22
4,Finance,98411.764706,17,18
5,HR,56214.285714,14,14
6,IT,131312.500000,16,18
7,Legal,123500.000000,14,15
8,Marketing,85769.230769,13,14
9,Operations,69714.285714,21,21


## 5. Data-quality checks

Missing salaries/ages and invalid ages are shown separately so that missing data is not confused with an invalid value.

In [15]:
missing_salary_query = """
SELECT employee_id, name, department, salary, age
FROM employees
WHERE salary IS NULL;
"""

missing_age_query = """
SELECT employee_id, name, department, salary, age
FROM employees
WHERE age IS NULL;
"""

invalid_age_query = """
SELECT employee_id, name, department, salary, age
FROM employees
WHERE age IS NOT NULL
  AND (
      age < 0
      OR age > 120
      OR age != CAST(age AS INTEGER)
  );
"""

print("Records with missing salaries:")
display(pd.read_sql_query(missing_salary_query, conn))

print("Records with missing ages:")
display(pd.read_sql_query(missing_age_query, conn))

print("Records with invalid ages (outside 0–120 or not a whole number):")
display(pd.read_sql_query(invalid_age_query, conn))

Records with missing salaries:


,employee_id,name,department,salary,age
0,1008,Tara Chavan,IT,None,35.0
1,1025,Aisha Malhotra,Legal,None,35.0
2,1050,Ishita Singh,IT,None,42.0
3,1074,Karan Khan,Engineering,None,46.0
4,1102,Anika Iyer,Sales,None,34.0
5,1129,Ishita Ghosh,Finance,None,54.0
6,1157,Sara Menon,Marketing,None,53.0
7,1189,Om Thomas,Engineering,None,42.0


Records with missing ages:


,employee_id,name,department,salary,age
0,1013,Anika Mehta,Engineering,187000.0,None
1,1039,Varun Pawar,Product,157000.0,None
2,1083,Kavya Bhat,Legal,160000.0,None
3,1120,Aarav Chavan,Operations,99000.0,None
4,1165,Zain Shetty,Customer Support,77000.0,None
5,1198,Tanvi Ansari,Customer Support,43000.0,None


Records with invalid ages (outside 0–120 or not a whole number):


,employee_id,name,department,salary,age
0,1032,Saanvi Mukherjee,Engineering,194000.0,-4.0
1,1097,Aarohi Dutta,Customer Support,46000.0,145.0
2,1144,Zoya Mehta,HR,41000.0,29.5


## 6. Pandas analysis

Record counts include every input row. Because the dataset contains a duplicate employee ID, unique-ID counts are also reported to avoid presenting record count as a confirmed headcount.

In [16]:
total_records = len(df)
unique_employee_ids = df["employee_id"].nunique(dropna=True)
average_salary = df["salary"].mean()

print(f"Total employee records: {total_records}")
print(f"Unique non-missing employee IDs: {unique_employee_ids}")
print(f"Overall average salary (non-missing values, all records): {average_salary:,.2f}")

print("\nEmployee records by department:")
department_counts = (
    df.groupby("department", dropna=False)
      .size()
      .reset_index(name="employee_records")
)
display(department_counts)

print("Unique employee IDs by department:")
unique_department_counts = (
    df.dropna(subset=["employee_id"])
      .groupby("department", dropna=False)["employee_id"]
      .nunique()
      .reset_index(name="unique_employee_ids")
)
display(unique_department_counts)

Total employee records: 200
Unique non-missing employee IDs: 200
Overall average salary (non-missing values, all records): 99,687.50

Employee records by department:


,department,employee_records
0,Administration,12
1,Customer Support,13
2,Data Science,15
3,Engineering,22
4,Finance,18
5,HR,14
6,IT,18
7,Legal,15
8,Marketing,14
9,Operations,21


Unique employee IDs by department:


,department,unique_employee_ids
0,Administration,12
1,Customer Support,13
2,Data Science,15
3,Engineering,22
4,Finance,18
5,HR,14
6,IT,18
7,Legal,15
8,Marketing,14
9,Operations,21


Management of Duplicate Records
- Find duplicates: Look for duplicate employee_id entries and duplicate rows.
- Remove duplicates: If the two rows have the same values, and one of them is known to be a duplicate, keep only one of the two records.
- Examine conflicted records: In case there is a duplicate employee ID with contradicting information, confirm the correct data with the source instead of simply deleting records.
- Verify after management: Count employee numbers and unique IDs to see whether the data is clean now.

Handling Missing Data
- Missing salaries: Ignore these data points when calculating salary average and the highest-paid employee. You should not use them since they can lead to errors, and do not fill them in with zeros.
- Missing ages: Report them and get their correct values from the source if possible.
- Missing department or employee ID: Try to find the right value from the source. The record without a unique ID cannot be considered a unique employee.
- Verify after handling missing values: Calculate statistics and show how many entries were fixed.
Our Shared Dataset
Our dataset has 200 entries where 8 salaries, 6 ages, 3 negative salaries and 3 invalid ages are missing. Employee IDs are unique with no duplicated rows.

## 7. Duplicate-record investigation

Duplicate IDs are flagged for review. Exact duplicate rows are shown separately from IDs that appear multiple times. No records are deleted automatically.

In [17]:
duplicate_id_mask = df.duplicated(subset=["employee_id"], keep=False)
duplicate_id_records = df.loc[duplicate_id_mask].sort_values("employee_id")

exact_duplicate_mask = df.duplicated(keep=False)
exact_duplicate_records = df.loc[exact_duplicate_mask].sort_values(
    REQUIRED_COLUMNS, na_position="last"
)

duplicate_id_count = int(df["employee_id"].duplicated().sum())
exact_duplicate_extra_rows = int(df.duplicated().sum())

print(f"Rows belonging to a repeated employee ID: {len(duplicate_id_records)}")
print(f"Repeated employee IDs beyond their first occurrence: {duplicate_id_count}")
print(f"Rows that are exact duplicates beyond their first occurrence: {exact_duplicate_extra_rows}")

print("\nAll rows with repeated employee IDs:")
display(duplicate_id_records)

print("Exact duplicate rows:")
display(exact_duplicate_records)

Rows belonging to a repeated employee ID: 0
Repeated employee IDs beyond their first occurrence: 0
Rows that are exact duplicates beyond their first occurrence: 0

All rows with repeated employee IDs:


,employee_id,name,department,salary,age,city,employment_type,job_level,years_experience


Exact duplicate rows:


,employee_id,name,department,salary,age,city,employment_type,job_level,years_experience


## 8. Validation checks

These assertions are regression checks for the supplied assessment dataset. If the CSV is intentionally changed, update the expected values after reviewing the new data.

In [19]:
# Validate the supplied 200-employee dataset.

assert len(df) == 200, f"Expected 200 records, got {len(df)}"

# Employee ID checks
assert df["employee_id"].nunique(dropna=True) == 200
assert df["employee_id"].isna().sum() == 0
assert df["employee_id"].duplicated().sum() == 0
assert df.duplicated().sum() == 0

# Missing-value checks
assert int(df["salary"].isna().sum()) == 8
assert int(df["age"].isna().sum()) == 6

# Invalid salary checks
negative_salary_count = int((df["salary"] < 0).sum())
assert negative_salary_count == 3

# Invalid age checks: negative, above 120, or fractional
invalid_age_mask = (
    df["age"].notna()
    & (
        (df["age"] < 0)
        | (df["age"] > 120)
        | (df["age"] % 1 != 0)
    )
)
invalid_age_count = int(invalid_age_mask.sum())
assert invalid_age_count == 3

# Salary calculations must exclude missing and negative salaries
valid_salary_mask = df["salary"].notna() & (df["salary"] >= 0)
valid_average_salary = df.loc[valid_salary_mask, "salary"].mean()

assert int(valid_salary_mask.sum()) == 189
assert round(float(valid_average_salary), 2) == 101349.21

# Query result checks
assert len(top_five) == 5
assert top_five["salary"].is_monotonic_decreasing
assert (top_five["salary"] >= 0).all()

assert department_averages["department"].is_monotonic_increasing

print("All validation checks passed.")

All validation checks passed.


## 9. Findings summary

This section generates the summary from the loaded data instead of relying on manually typed figures.

In [20]:
# Calculate data-quality counts
missing_salary_count = int(df["salary"].isna().sum())
missing_age_count = int(df["age"].isna().sum())

invalid_age_count = int(
    (
        df["age"].notna()
        & (
            (df["age"] < 0)
            | (df["age"] > 120)
            | (df["age"] % 1 != 0)
        )
    ).sum()
)

negative_salary_count = int((df["salary"] < 0).sum())

duplicate_id_count = int(df["employee_id"].duplicated().sum())
exact_duplicate_extra_rows = int(df.duplicated().sum())

# Use only valid salaries for salary-based calculations
valid_salary_mask = df["salary"].notna() & (df["salary"] >= 0)
valid_average_salary = df.loc[valid_salary_mask, "salary"].mean()

# Count employee records and unique IDs separately
total_records = len(df)
unique_employee_ids = df["employee_id"].nunique(dropna=True)

print("TASK 2 — FINDINGS SUMMARY")
print("-" * 36)

print(f"Total employee records: {total_records}")
print(f"Unique non-missing employee IDs: {unique_employee_ids}")
print(f"Average valid salary: {valid_average_salary:,.2f}")

print("\nData quality:")
print(f"Missing salaries: {missing_salary_count}")
print(f"Missing ages: {missing_age_count}")
print(f"Invalid ages: {invalid_age_count}")
print(f"Negative salaries: {negative_salary_count}")
print(f"Repeated employee IDs beyond first occurrence: {duplicate_id_count}")
print(f"Exact duplicate rows beyond first occurrence: {exact_duplicate_extra_rows}")

print("\nTop-paid employee:")
if not top_five.empty:
    top_employee = top_five.iloc[0]
    print(
        f"{top_employee['name']} "
        f"(ID {int(top_employee['employee_id'])}), "
        f"salary {top_employee['salary']:,.2f}"
    )
else:
    print("No valid salary records are available.")

print("\nDepartment average salaries:")
display(
    department_averages[
        ["department", "average_salary"]
    ]
)

TASK 2 — FINDINGS SUMMARY
------------------------------------
Total employee records: 200
Unique non-missing employee IDs: 200
Average valid salary: 101,349.21

Data quality:
Missing salaries: 8
Missing ages: 6
Invalid ages: 3
Negative salaries: 3
Repeated employee IDs beyond first occurrence: 0
Exact duplicate rows beyond first occurrence: 0

Top-paid employee:
Rahul Ansari (ID 1119), salary 221,000.00

Department average salaries:


,department,average_salary
0,Administration,58583.333333
1,Customer Support,56692.307692
2,Data Science,155266.666667
3,Engineering,126050.000000
4,Finance,98411.764706
5,HR,56214.285714
6,IT,131312.500000
7,Legal,123500.000000
8,Marketing,85769.230769
9,Operations,69714.285714
